In [1]:
# generate_splits_from_xes_with_resource_parallel.py
import os
import math
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm import tqdm

# pm4py for XES parsing
from pm4py.objects.log.importer.xes import importer as xes_importer
from pm4py.objects.conversion.log import converter as log_converter

from framework.utils import numeric_case_id_series

# ---------- I/O & normalization ----------

def load_xes_to_df(xes_path: str) -> pd.DataFrame:
    """
    Load a XES log -> normalized pandas DataFrame with columns:
    ['process','timestamp_abs','case_id','activity','resource','status'].
    - timestamp_abs in HOURS (float) for stability across logs with big ranges
    - process is file stem
    - status is 'running'
    """
    log = xes_importer.apply(xes_path)
    df = log_converter.apply(log, variant=log_converter.Variants.TO_DATA_FRAME)

    case_col = "case:concept:name"
    act_col = "concept:name"
    time_col = "time:timestamp"
    res_col = None
    for c in ["org:resource", "Resource", "resource"]:
        if c in df.columns:
            res_col = c
            break

    if case_col not in df.columns or act_col not in df.columns or time_col not in df.columns:
        raise ValueError(f"Missing required XES columns in {xes_path}")

    proc = Path(xes_path).stem
    out = pd.DataFrame()
    out["process"] = proc
    out["case_id_raw"] = df[case_col].astype(str)
    out["case_id"] = numeric_case_id_series(out["case_id_raw"]).astype(str)
    out["activity"] = df[act_col].astype(str)
    # ns -> hours (1 hour = 3.6e12 ns)
    out["timestamp_abs"] = pd.to_datetime(df[time_col]).astype("int64") / 3_600_000_000_000
    out["resource"] = (df[res_col].astype(str) if res_col else "").fillna("")
    out["status"] = "running"

    out = out.sort_values(["case_id", "timestamp_abs"], kind="mergesort").reset_index(drop=True)
    return out


def build_vocab_activity(df_all: pd.DataFrame):
    acts = sorted(df_all["activity"].unique())
    activity_to_int = {a: i + 1 for i, a in enumerate(acts)}  # 0 = padding
    int_to_activity = {i: a for a, i in activity_to_int.items()}
    return activity_to_int, int_to_activity


def build_vocab_process(df_all: pd.DataFrame):
    procs = sorted(df_all["process"].unique())
    process_to_int = {p: i + 1 for i, p in enumerate(procs)}  # 0 = padding
    int_to_process = {i: p for p, i in process_to_int.items()}
    return process_to_int, int_to_process

def test_only_same_resource_parallel_ge(df_test_prefix: pd.DataFrame, thresh: float) -> pd.DataFrame:
    """
    Keep prefixes with shared_resource_overlap_ratio_max >= thresh.
    (thresh is a float in [0,1], e.g., 0.40 or 0.70)
    """
    return df_test_prefix[df_test_prefix["shared_resource_overlap_ratio_max"] >= thresh].reset_index(drop=True)

def test_only_same_resource_parallel_ge_40(df_test_prefix: pd.DataFrame) -> pd.DataFrame:
    return test_only_same_resource_parallel_ge(df_test_prefix, 0.40)

def test_only_same_resource_parallel_ge_70(df_test_prefix: pd.DataFrame) -> pd.DataFrame:
    return test_only_same_resource_parallel_ge(df_test_prefix, 0.70)


# ---------- concurrency / intervals ----------

def build_case_intervals(df_events: pd.DataFrame):
    """
    Per-case [start, end] in absolute time units (hours).
    """
    g = df_events.groupby("case_id")
    starts = g["timestamp_abs"].min().astype(float)
    ends = g["timestamp_abs"].max().astype(float)
    case_ids = starts.index.tolist()
    arr_case_ids = np.array(case_ids)
    arr_starts = starts.values
    arr_ends = ends.values
    idx_of = {cid: i for i, cid in enumerate(case_ids)}
    return arr_case_ids, arr_starts, arr_ends, idx_of


def build_case_resources(df_events: pd.DataFrame):
    """
    Map each case_id -> set(resources) appearing in that case (empty if none).
    """
    def _to_set(s):
        return set(r for r in s if isinstance(r, str) and len(r) > 0)
    g = df_events.groupby("case_id")["resource"].apply(list)
    return {cid: _to_set(lst) for cid, lst in g.items()}


def active_case_ids_at_time(case_ids, starts, ends, idx_of, case_id, t_abs):
    """
    Return numpy array of case_ids active at t_abs, excluding `case_id`.
    Active if start <= t_abs < end.
    """
    mask = (starts <= t_abs) & (ends > t_abs)
    if case_id in idx_of:
        i = idx_of[case_id]
        if 0 <= i < mask.shape[0]:
            mask[i] = False
    return case_ids[mask]


def has_parallel_at_time(case_ids, starts, ends, idx_of, case_id, t_abs) -> bool:
    return active_case_ids_at_time(case_ids, starts, ends, idx_of, case_id, t_abs).size > 0


def has_shared_resource_parallel_at_time(case_ids, starts, ends, idx_of, case_resources,
                                         case_id, t_abs) -> bool:
    """
    NEW semantics for TEST_SAMERES_PAR:
    True if, at time t_abs, there exists ANOTHER active case whose resource-set
    intersects with the target case's resource-set (i.e., cases share ≥1 resource).
    Overlap in time is required via active_case_ids_at_time(...).
    """
    my_res = case_resources.get(case_id, set())
    if not my_res:
        return False
    active_ids = active_case_ids_at_time(case_ids, starts, ends, idx_of, case_id, t_abs)
    for ocid in active_ids:
        if my_res & case_resources.get(ocid, set()):
            return True
    return False


# ---------- prefix generation ----------

def generate_prefix_rows(df_events: pd.DataFrame) -> pd.DataFrame:
    """
    From event rows (single process), generate per-prefix rows.
    Emits ONLY prefixes with length in [2, L-1]  (i.e., skips single-prefix and full-prefix).
    Adds flags:
      - has_parallel
      - has_same_resource_parallel  (≥1 shared resource + overlap)
      - has_same_resource_parallel_ge_40 / _ge_70
    Also stores:
      - shared_resource_overlap_ratio_max
    """
    g = df_events.groupby("case_id", sort=False)
    case_end = g["timestamp_abs"].max().to_dict()
    case_start = g["timestamp_abs"].min().to_dict()

    # global structures
    arr_case_ids, arr_starts, arr_ends, idx_of = build_case_intervals(df_events)
    case_resources = build_case_resources(df_events)

    rows = []
    for cid, grp in g:
        grp = grp.sort_values("timestamp_abs")
        proc = grp["process"].iloc[0]
        acts = grp["activity"].tolist()
        times_abs = grp["timestamp_abs"].to_numpy(dtype=float)

        cstart = case_start[cid]; cend = case_end[cid]
        times_rel = times_abs - cstart

        L = len(acts)
        if L <= 2:
            # For cases with ≤2 events, there is no prefix with length in [2, L-1]
            continue

        # k = 2..L-1  (skip single prefix and full prefix)
        for k in range(2, L):
            t_abs = float(times_abs[k - 1])
            remaining = float(cend - t_abs)
            # defensive (should be >0 because k<L)
            REM_EPS = 1e-9
            if remaining <= REM_EPS:
                continue

            prefix = acts[:k]
            t_rel = float(times_rel[k - 1])
            next_act = acts[k]  # safe: k < L

            has_par = has_parallel_at_time(arr_case_ids, arr_starts, arr_ends, idx_of, cid, t_abs)

            has_same_res_par = has_shared_resource_parallel_at_time(
                arr_case_ids, arr_starts, arr_ends, idx_of, case_resources, cid, t_abs
            )

            # compute max overlap ratio and thresholded flags
            my_res = case_resources.get(cid, set())
            overlap_max = 0.0
            if my_res:
                denom = len(my_res)
                active_ids = active_case_ids_at_time(arr_case_ids, arr_starts, arr_ends, idx_of, cid, t_abs)
                for ocid in active_ids:
                    o_res = case_resources.get(ocid, set())
                    if not o_res:
                        continue
                    inter = my_res & o_res
                    overlap_max = max(overlap_max, len(inter) / denom)

            has_same_res_par_ge_40 = overlap_max >= 0.40
            has_same_res_par_ge_70 = overlap_max >= 0.70

            rows.append({
                "process": proc,
                "case_id": cid,
                "prefix": prefix,
                "list_timestamps_case": list(times_rel[:k].astype(float)),
                "remaining_time": remaining,
                "next_activity": next_act,
                "prefix_time": t_rel,
                "prefix_abs_time": t_abs,
                "has_parallel": has_par,
                "has_same_resource_parallel": has_same_res_par,
                "shared_resource_overlap_ratio_max": overlap_max,
                "has_same_resource_parallel_ge_40": has_same_res_par_ge_40,
                "has_same_resource_parallel_ge_70": has_same_res_par_ge_70,
            })

    return pd.DataFrame(rows)




# ---------- splits ----------

def temporal_case_split(df_events: pd.DataFrame, ratios=(0.6, 0.2, 0.2)):
    g = df_events.groupby("case_id")["timestamp_abs"].min().sort_values()
    case_ids = g.index.to_list()
    n = len(case_ids)
    n_tr = int(n * ratios[0])
    n_va = int(n * ratios[1])
    tr_cases = set(case_ids[:n_tr])
    va_cases = set(case_ids[n_tr:n_tr + n_va])
    te_cases = set(case_ids[n_tr + n_va:])

    tr = df_events[df_events["case_id"].isin(tr_cases)].copy()
    va = df_events[df_events["case_id"].isin(va_cases)].copy()
    te = df_events[df_events["case_id"].isin(te_cases)].copy()
    return tr, va, te


def encode_prefix_df(df_prefix: pd.DataFrame,
                     activity_to_int: dict,
                     process_to_int: dict) -> pd.DataFrame:
    def enc_list(lst):
        return [activity_to_int.get(a, 0) for a in lst]
    df = df_prefix.copy()
    df["prefix_int"] = df["prefix"].apply(enc_list)
    df["next_activity_int"] = df["next_activity"].map(lambda a: activity_to_int.get(a, 0) if a is not None else 0)
    df["process_int"] = df["process"].map(lambda p: process_to_int.get(p, 0))
    return df


# ---------- test-set filters ----------

def test_only_parallel(df_test_prefix: pd.DataFrame) -> pd.DataFrame:
    return df_test_prefix[df_test_prefix["has_parallel"]].reset_index(drop=True)

def test_only_no_parallel(df_test_prefix: pd.DataFrame) -> pd.DataFrame:
    return df_test_prefix[~df_test_prefix["has_parallel"]].reset_index(drop=True)

def test_only_same_resource_parallel(df_test_prefix: pd.DataFrame) -> pd.DataFrame:
    """
    Uses UPDATED semantics set in generate_prefix_rows:
    keep prefixes where the case shares ≥1 resource with at least one
    concurrently active case at the prefix time.
    """
    return df_test_prefix[df_test_prefix["has_same_resource_parallel"]].reset_index(drop=True)

# def make_gen_split(train_prefix: pd.DataFrame, val_prefix: pd.DataFrame, test_prefix: pd.DataFrame):
#     train_pref_set = set(train_prefix["prefix"].apply(tuple).tolist())
#     test_gen = test_prefix[~test_prefix["prefix"].apply(tuple).isin(train_pref_set)].reset_index(drop=True)
#     sampled_prefixes = set(test_gen["prefix"].apply(tuple).tolist())
#     def _clean(df):
#         return df[~df["prefix"].apply(tuple).isin(sampled_prefixes)].reset_index(drop=True)
#     train_gen = _clean(train_prefix)
#     val_gen = _clean(val_prefix)
#     return train_gen, val_gen, test_gen


def make_gen_split(train_prefix: pd.DataFrame,
                   val_prefix: pd.DataFrame,
                   test_prefix: pd.DataFrame,
                   test_ratio: float = 0.20,
                   seed: int = 42):
    """
    Build a GEN split where:
      - GEN test is ~test_ratio of ALL rows across (train+val+test),
      - Only the EXACT sampled rows are removed from TRAIN and VAL,
      - No filtering by 'prefix in train' (to guarantee enough candidates).
    """
    # Copy + tag origin + stable row ids
    tr = train_prefix.copy(); tr["_origin"] = "train"; tr["_rid"] = np.arange(len(tr))
    va = val_prefix.copy();   va["_origin"] = "val";   va["_rid"] = np.arange(len(va))
    te = test_prefix.copy();  te["_origin"] = "test";  te["_rid"] = np.arange(len(te))

    all_df = pd.concat([tr, va, te], ignore_index=True)

    total_n = len(all_df)
    target_n = int(math.ceil(total_n * test_ratio))

    # Sample from ALL rows (train+val+test) to hit the 20% target
    test_gen = all_df.sample(n=min(target_n, total_n), random_state=seed, replace=False)

    # Figure out which exact rows came from which split
    train_drop = set(test_gen.loc[test_gen["_origin"] == "train", "_rid"])
    val_drop   = set(test_gen.loc[test_gen["_origin"] == "val",   "_rid"])
    # (Rows sampled from original TEST don't require removals from train/val)

    # Remove ONLY those exact sampled rows from train/val
    train_gen = tr[~tr["_rid"].isin(train_drop)].drop(columns=["_origin", "_rid"]).reset_index(drop=True)
    val_gen   = va[~va["_rid"].isin(val_drop)].drop(columns=["_origin", "_rid"]).reset_index(drop=True)

    # Finalize GEN test (drop helper cols)
    test_gen  = test_gen.drop(columns=["_origin", "_rid"]).reset_index(drop=True)

    return train_gen, val_gen, test_gen

def has_shared_resource_parallel_frac_at_time(case_ids, starts, ends, idx_of, case_resources,
                                              case_id, t_abs, threshold: float) -> bool:
    """
    True if, at time t_abs, there exists ANOTHER active case whose resource-set
    overlaps with the target case's resource-set by >= `threshold` of the target's resources.
    Overlap ratio = |R_my ∩ R_other| / |R_my|. Empty R_my -> False.
    """
    my_res = case_resources.get(case_id, set())
    if not my_res:
        return False
    denom = len(my_res)
    active_ids = active_case_ids_at_time(case_ids, starts, ends, idx_of, case_id, t_abs)
    for ocid in active_ids:
        o_res = case_resources.get(ocid, set())
        if not o_res:
            continue
        inter = my_res & o_res
        if len(inter) / denom >= threshold:
            return True
    return False





# ---------- per-log pipeline ----------

def process_one_log(xes_path: str, out_dir: str, ratios=(0.6, 0.2, 0.2)):
    os.makedirs(out_dir, exist_ok=True)
    df_events = load_xes_to_df(xes_path)
    name = Path(xes_path).stem

    # vocab on the whole log
    activity_to_int, int_to_activity = build_vocab_activity(df_events)
    process_to_int, int_to_process = build_vocab_process(df_events)

    # save vocabs
    with open(f"{out_dir}/{name}_activity_to_int.p", "wb") as f: pickle.dump(activity_to_int, f)
    with open(f"{out_dir}/{name}_int_to_activity.p", "wb") as f: pickle.dump(int_to_activity, f)
    with open(f"{out_dir}/{name}_process_to_int.p", "wb") as f: pickle.dump(process_to_int, f)
    with open(f"{out_dir}/{name}_int_to_process.p", "wb") as f: pickle.dump(int_to_process, f)

    # temporal case split on raw events
    tr_e, va_e, te_e = temporal_case_split(df_events, ratios)

    # generate prefixes per split
    print(f"[{name}] generating prefixes (STD base splits)...")
    tr_pfx = generate_prefix_rows(tr_e)
    va_pfx = generate_prefix_rows(va_e)
    te_pfx = generate_prefix_rows(te_e)

    # encode ints
    tr_pfx_enc = encode_prefix_df(tr_pfx, activity_to_int, process_to_int)
    va_pfx_enc = encode_prefix_df(va_pfx, activity_to_int, process_to_int)
    te_pfx_enc = encode_prefix_df(te_pfx, activity_to_int, process_to_int)

    # ---------------- STD ----------------
    tr_pfx_enc.to_pickle(f"{out_dir}/{name}_STD_train_prefix.pkl")
    va_pfx_enc.to_pickle(f"{out_dir}/{name}_STD_val_prefix.pkl")
    te_pfx_enc.to_pickle(f"{out_dir}/{name}_STD_test_prefix.pkl")
    print(f"[{name}] STD: train={len(tr_pfx_enc)} val={len(va_pfx_enc)} test={len(te_pfx_enc)}")

    # ---------------- TEST_PAR ----------------
    te_par = test_only_parallel(te_pfx_enc)
    te_par.to_pickle(f"{out_dir}/{name}_TEST_PAR_test_prefix.pkl")
    print(f"[{name}] TEST_PAR: test={len(te_par)} (train/val = STD)")

    # ---------------- TEST_NOPAR ----------------
    te_nopar = test_only_no_parallel(te_pfx_enc)
    te_nopar.to_pickle(f"{out_dir}/{name}_TEST_NOPAR_test_prefix.pkl")
    print(f"[{name}] TEST_NOPAR: test={len(te_nopar)} (train/val = STD)")

    # ---------------- TEST_SAMERES_PAR (UPDATED SEMANTICS) ----------------
    te_sameres = test_only_same_resource_parallel(te_pfx_enc)
    te_sameres.to_pickle(f"{out_dir}/{name}_TEST_SAMERES_PAR_test_prefix.pkl")
    print(f"[{name}] TEST_SAMERES_PAR: test={len(te_sameres)} (train/val = STD)")

    # ---------------- GEN ----------------
    tr_gen, va_gen, te_gen = make_gen_split(tr_pfx_enc, va_pfx_enc, te_pfx_enc)
    tr_gen.to_pickle(f"{out_dir}/{name}_GEN_train_prefix.pkl")
    va_gen.to_pickle(f"{out_dir}/{name}_GEN_val_prefix.pkl")
    te_gen.to_pickle(f"{out_dir}/{name}_GEN_test_prefix.pkl")
    print(f"[{name}] GEN: train={len(tr_gen)} val={len(va_gen)} test={len(te_gen)}")

    # ---------------- TEST_SAMERES40_PAR ----------------
    te_sameres40 = test_only_same_resource_parallel_ge_40(te_pfx_enc)
    te_sameres40.to_pickle(f"{out_dir}/{name}_TEST_SAMERES40_PAR_test_prefix.pkl")
    print(f"[{name}] TEST_SAMERES40_PAR: test={len(te_sameres40)} (train/val = STD)")

    # ---------------- TEST_SAMERES70_PAR ----------------
    te_sameres70 = test_only_same_resource_parallel_ge_70(te_pfx_enc)
    te_sameres70.to_pickle(f"{out_dir}/{name}_TEST_SAMERES70_PAR_test_prefix.pkl")
    print(f"[{name}] TEST_SAMERES70_PAR: test={len(te_sameres70)} (train/val = STD)")


    return {
        "name": name,
        "STD": (len(tr_pfx_enc), len(va_pfx_enc), len(te_pfx_enc)),
        "TEST_PAR": len(te_par),
        "TEST_NOPAR": len(te_nopar),
        "TEST_SAMERES_PAR": len(te_sameres),
        "TEST_SAMERES40_PAR": len(te_sameres40),   # NEW
        "TEST_SAMERES70_PAR": len(te_sameres70),   # NEW
        "GEN": (len(tr_gen), len(va_gen), len(te_gen)),
    }



# ---------- driver ----------

def main():
    raw_dir = "raw_dataset"
    out_dir = "dataset_xes"
    os.makedirs(out_dir, exist_ok=True)

    xes_files = sorted([str(p) for p in Path(raw_dir).glob("*.xes")])
    if not xes_files:
        print(f"No .xes files found in {raw_dir}")
        return

    summary = []
    for xp in xes_files:
        try:
            info = process_one_log(xp, out_dir=out_dir, ratios=(0.6, 0.2, 0.2))
            summary.append(info)
        except Exception as e:
            print(f"[ERROR] {xp}: {e}")

    rows = []
    for s in summary:
        rows.append({
            "log": s["name"],
            "STD_train": s["STD"][0],
            "STD_val": s["STD"][1],
            "STD_test": s["STD"][2],
            "TEST_PAR_test": s["TEST_PAR"],
            "TEST_NOPAR_test": s["TEST_NOPAR"],
            "TEST_SAMERES_PAR_test": s["TEST_SAMERES_PAR"],
            "TEST_SAMERES40_PAR_test": s["TEST_SAMERES40_PAR"],  # NEW
            "TEST_SAMERES70_PAR_test": s["TEST_SAMERES70_PAR"],  # NEW
            "GEN_train": s["GEN"][0],
            "GEN_val": s["GEN"][1],
            "GEN_test": s["GEN"][2],
        })

    if rows:
        pd.DataFrame(rows).to_csv(os.path.join(out_dir, "summary.csv"), index=False)
        print(f"Saved summary → {os.path.join(out_dir, 'summary.csv')}")


if __name__ == "__main__":
    main()


c:\Users\kiran.busch\anaconda3\envs\muliple-predictive-process-monitoring\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
parsing log, completed traces :: 100%|██████████| 1199/1199 [00:06<00:00, 184.51it/s]


[BPIC15_1] generating prefixes (STD base splits)...
[BPIC15_1] STD: train=27688 val=11496 test=9520
[BPIC15_1] TEST_PAR: test=9520 (train/val = STD)
[BPIC15_1] TEST_NOPAR: test=0 (train/val = STD)
[BPIC15_1] TEST_SAMERES_PAR: test=9520 (train/val = STD)
[BPIC15_1] GEN: train=22216 val=9157 test=9741
[BPIC15_1] TEST_SAMERES40_PAR: test=9520 (train/val = STD)
[BPIC15_1] TEST_SAMERES70_PAR: test=9116 (train/val = STD)


parsing log, completed traces :: 100%|██████████| 832/832 [00:04<00:00, 180.74it/s]


[BPIC15_2] generating prefixes (STD base splits)...
[BPIC15_2] STD: train=25098 val=9689 test=7502
[BPIC15_2] TEST_PAR: test=7502 (train/val = STD)
[BPIC15_2] TEST_NOPAR: test=0 (train/val = STD)
[BPIC15_2] TEST_SAMERES_PAR: test=7469 (train/val = STD)
[BPIC15_2] GEN: train=20035 val=7802 test=8458
[BPIC15_2] TEST_SAMERES40_PAR: test=7434 (train/val = STD)
[BPIC15_2] TEST_SAMERES70_PAR: test=7164 (train/val = STD)


parsing log, completed traces :: 100%|██████████| 1409/1409 [00:06<00:00, 212.17it/s]


...

[ERROR] raw_dataset\Sepsis.xes: 'str' object has no attribute 'fillna'


parsing log, completed traces :: 100%|██████████| 150370/150370 [00:57<00:00, 2605.04it/s]


[Traffic_Fines] generating prefixes (STD base splits)...
[Traffic_Fines] STD: train=240649 val=41674 test=36170
[Traffic_Fines] TEST_PAR: test=36170 (train/val = STD)
[Traffic_Fines] TEST_NOPAR: test=0 (train/val = STD)
[Traffic_Fines] TEST_SAMERES_PAR: test=36170 (train/val = STD)
[Traffic_Fines] GEN: train=192477 val=33354 test=63699
[Traffic_Fines] TEST_SAMERES40_PAR: test=36170 (train/val = STD)
[Traffic_Fines] TEST_SAMERES70_PAR: test=36045 (train/val = STD)


parsing log, completed traces :: 100%|██████████| 1434/1434 [00:00<00:00, 2031.16it/s]


[env_permit] generating prefixes (STD base splits)...
[env_permit] STD: train=3641 val=1132 test=1052
[env_permit] TEST_PAR: test=914 (train/val = STD)
[env_permit] TEST_NOPAR: test=138 (train/val = STD)
[env_permit] TEST_SAMERES_PAR: test=215 (train/val = STD)
[env_permit] GEN: train=2908 val=913 test=1165
[env_permit] TEST_SAMERES40_PAR: test=211 (train/val = STD)
[env_permit] TEST_SAMERES70_PAR: test=176 (train/val = STD)
Saved summary → dataset_xes\summary.csv


In [ ]:
import math
from pathlib import Path

import numpy as np
import pandas as pd

def compute_log_stats(df_events: pd.DataFrame, scenario_name: str) -> dict:
    """
    Compute event log statistics for one preprocessed event log.

    Assumes df_events has the columns produced by load_xes_to_df:
    ['process', 'timestamp_abs', 'case_id', 'activity', 'resource', 'status']

    CT (cycle time) is computed in HOURS (since timestamp_abs is in hours).
    CL (case length) is the number of events in a case.
    """
    # group by case
    g = df_events.groupby("case_id")

    # # Traces
    n_traces = g.ngroups

    # # Unique traces (unique activity sequences)
    # Each trace is a tuple of activities for that case
    trace_seqs = g["activity"].apply(tuple)
    n_unique_traces = trace_seqs.nunique()

    # # Activities (distinct activity labels)
    n_activities = df_events["activity"].nunique()

    # # Resources (distinct non-empty resources)
    res_series = df_events["resource"].replace("", np.nan)
    n_resources = res_series.nunique(dropna=True)

    # CT: per-case cycle time = max(timestamp_abs) - min(timestamp_abs)
    ct_series = g["timestamp_abs"].max() - g["timestamp_abs"].min()
    min_ct = ct_series.min()
    mean_ct = ct_series.mean()
    max_ct = ct_series.max()
    sd_ct = ct_series.std(ddof=1)  # sample standard deviation, like most stats packages

    # CL: per-case length (#events)
    cl_series = g.size()
    min_cl = cl_series.min()
    mean_cl = cl_series.mean()
    max_cl = cl_series.max()

    return {
        "Evaluation scenario": scenario_name,
        "# Traces": int(n_traces),
        "# Unique traces": int(n_unique_traces),
        "# Activities": int(n_activities),
        #"# Resources": int(n_resources),
        #"Min CT": float(min_ct),
        "Mean CT": float(mean_ct),
        "Max CT": float(max_ct),
        "SD CT": float(sd_ct),
        #"Min CL": int(min_cl),
        "Mean CL": float(mean_cl),
        "Max CL": int(max_cl),
    }


raw_dir = Path("raw_dataset/paper")

xes_files = sorted(raw_dir.glob("*.xes"))
if not xes_files:
    print(f"No .xes files found in {raw_dir}")

rows = []
for xes_path in xes_files:
    print(f"Processing {xes_path} ...")
    # Use your existing preprocessing to get a normalized df
    df_events = load_xes_to_df(str(xes_path))

    # Scenario name: here we just use the file stem; adapt if you want
    scenario_name = xes_path.stem

    stats_row = compute_log_stats(df_events, scenario_name)
    rows.append(stats_row)

df_stats = pd.DataFrame(rows)



Processing raw_dataset\paper\BPI_Challenge_2012.xes ...


parsing log, completed traces :: 100%|██████████| 13087/13087 [00:18<00:00, 702.59it/s]


Processing raw_dataset\paper\BPIC15_1.xes ...


parsing log, completed traces :: 100%|██████████| 1199/1199 [00:08<00:00, 137.90it/s]


Processing raw_dataset\paper\BPIC15_2.xes ...


...

Processing raw_dataset\paper\BPIC20_DomesticDeclarations.xes ...


parsing log, completed traces :: 100%|██████████| 10500/10500 [00:04<00:00, 2161.87it/s]


Processing raw_dataset\paper\BPIC20_InternationalDeclarations.xes ...


parsing log, completed traces :: 100%|██████████| 6449/6449 [00:06<00:00, 961.21it/s] 


KeyError: "['Min CT'] not in index"

In [12]:
# Optional: round numeric columns to 2 decimals for nicer display
numeric_cols = [
    "Mean CT", "Max CT", "SD CT",
    "Mean CL"
]
df_stats[numeric_cols] = df_stats[numeric_cols].round(2)

print("\nEvent log statistics:")
print(df_stats)

# Optional: save to CSV
out_path = raw_dir / "log_stats.csv"
df_stats.to_csv(out_path, index=False)
print(f"\nSaved statistics to {out_path}")

df_stats


Event log statistics:
                Evaluation scenario  # Traces  # Unique traces  # Activities  \
0                BPI_Challenge_2012     13087             4366            24   
1                          BPIC15_1      1199             1170           398   
2                          BPIC15_2       832              828           410   
3                          BPIC15_3      1409             1349           383   
4                          BPIC15_4      1053             1049           356   
5       BPIC20_DomesticDeclarations     10500               99            17   
6  BPIC20_InternationalDeclarations      6449              753            34   

   Mean CT    Max CT    SD CT  Mean CL  Max CL  
0   206.92   3292.32   291.04    20.04     175  
1  2297.29  35664.00  2913.73    43.55     101  
2  3842.50  31824.00  4044.36    53.31     132  
3  1493.55  36288.00  2343.54    42.36     124  
4  2803.33  22248.00  2597.33    44.91     116  
5   276.61  11262.67   408.48     5.37    

,Evaluation scenario,# Traces,# Unique traces,# Activities,Mean CT,Max CT,SD CT,Mean CL,Max CL
0,BPI_Challenge_2012,13087,4366,24,206.92,3292.32,291.04,20.04,175
1,BPIC15_1,1199,1170,398,2297.29,35664.00,2913.73,43.55,101
2,BPIC15_2,832,828,410,3842.50,31824.00,4044.36,53.31,132
3,BPIC15_3,1409,1349,383,1493.55,36288.00,2343.54,42.36,124
4,BPIC15_4,1053,1049,356,2803.33,22248.00,2597.33,44.91,116
5,BPIC20_DomesticDeclarations,10500,99,17,276.61,11262.67,408.48,5.37,24
6,BPIC20_InternationalDeclarations,6449,753,34,2074.92,17808.00,1880.42,11.19,27


In [15]:
df_stats.to_latex()

'\\begin{tabular}{llrrrrrrrr}\n\\toprule\n & Evaluation scenario & # Traces & # Unique traces & # Activities & Mean CT & Max CT & SD CT & Mean CL & Max CL \\\\\n\\midrule\n0 & BPI_Challenge_2012 & 13087 & 4366 & 24 & 206.920000 & 3292.320000 & 291.040000 & 20.040000 & 175 \\\\\n1 & BPIC15_1 & 1199 & 1170 & 398 & 2297.290000 & 35664.000000 & 2913.730000 & 43.550000 & 101 \\\\\n2 & BPIC15_2 & 832 & 828 & 410 & 3842.500000 & 31824.000000 & 4044.360000 & 53.310000 & 132 \\\\\n3 & BPIC15_3 & 1409 & 1349 & 383 & 1493.550000 & 36288.000000 & 2343.540000 & 42.360000 & 124 \\\\\n4 & BPIC15_4 & 1053 & 1049 & 356 & 2803.330000 & 22248.000000 & 2597.330000 & 44.910000 & 116 \\\\\n5 & BPIC20_DomesticDeclarations & 10500 & 99 & 17 & 276.610000 & 11262.670000 & 408.480000 & 5.370000 & 24 \\\\\n6 & BPIC20_InternationalDeclarations & 6449 & 753 & 34 & 2074.920000 & 17808.000000 & 1880.420000 & 11.190000 & 27 \\\\\n\\bottomrule\n\\end{tabular}\n'

In [21]:
df_events = load_xes_to_df("raw_dataset/BPI_Challenge_2012.xes")

parsing log, completed traces :: 100%|██████████| 13087/13087 [00:18<00:00, 707.31it/s]


In [22]:
df_events

,process,case_id_raw,case_id,activity,timestamp_abs,resource,status
0,NaN,173688,173688,A_SUBMITTED,365952.645707,112,running
1,NaN,173688,173688,A_PARTLYSUBMITTED,365952.645800,112,running
2,NaN,173688,173688,A_PREACCEPTED,365952.660529,112,running
3,NaN,173688,173688,W_Completeren aanvraag,365952.660799,112,running
4,NaN,173688,173688,W_Completeren aanvraag,365963.612899,nan,running
...,...,...,...,...,...,...,...
262195,NaN,214376,214376,A_PARTLYSUBMITTED,369599.854840,112,running
262196,NaN,214376,214376,W_Afhandelen leads,369599.867024,112,running
262197,NaN,214376,214376,W_Afhandelen leads,369609.446316,11169,running
262198,NaN,214376,214376,A_DECLINED,369609.460311,11169,running
